In [16]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import accuracy_score

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.impute import SimpleImputer, KNNImputer
import category_encoders as ce

from sklearn.pipeline import Pipeline
from sklearn import neighbors
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

In [17]:
variables_entrada = ['age', 'gender', 'daily_gaming_hours', 'game_genre', 'sleep_hours', 'sleep_quality',
       'sleep_disruption_frequency', 'academic_work_performance', 'grades_gpa',
       'work_productivity_score', 'mood_state', 'mood_swing_frequency',
       'withdrawal_symptoms', 'loss_of_other_interests',
       'continued_despite_problems', 'eye_strain', 'back_neck_pain',
       'weight_change_kg', 'exercise_hours_weekly', 'social_isolation_score',
       'face_to_face_social_hours_weekly', 'monthly_game_spending_usd',
       'years_gaming']
variable_salida = 'gaming_addiction_risk_level'

datos = pd.read_csv('Gaming and Mental Health.csv', delimiter=',', header=0)

X = datos.loc[:, variables_entrada].copy()
y = datos.loc[:, variable_salida].copy()

In [18]:
X.columns.size

23

#### Variables numéricas:

que mierda significa tener un work_productivity_score de 6 ???? es todo lo subjetivo

que mierda es el social_isolation_score y en que se basa

grades_gpa va de 1 a 4, puta mierda de americanadas

In [19]:
X.describe()

,age,daily_gaming_hours,sleep_hours,grades_gpa,work_productivity_score,weight_change_kg,exercise_hours_weekly,social_isolation_score,face_to_face_social_hours_weekly,monthly_game_spending_usd,years_gaming
count,1000.000000,1000.000000,1000.000000,754.000000,674.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,20.475000,6.151400,5.738100,2.518037,5.394659,1.513400,6.945900,3.872000,7.654500,105.219730,5.796000
std,4.116105,2.867194,1.441213,0.872312,2.898742,1.432212,1.805027,2.091409,3.751954,113.886768,3.775532
min,13.000000,0.500000,3.000000,1.010000,1.000000,0.000000,0.700000,1.000000,0.000000,0.100000,1.000000
25%,18.000000,4.100000,4.800000,1.760000,3.000000,0.400000,5.700000,2.000000,5.000000,32.592500,3.000000
50%,20.000000,6.000000,5.700000,2.530000,5.000000,1.100000,7.000000,4.000000,8.000000,66.405000,5.000000
75%,22.000000,8.025000,6.600000,3.280000,8.000000,2.100000,8.200000,5.000000,10.400000,126.242500,8.000000
max,35.000000,15.100000,9.000000,4.000000,10.000000,8.900000,11.500000,10.000000,16.700000,499.270000,20.000000


#### Variables categóricas:

Para sleep_quality, sleep_disruption_frequency, academic_work_performance y mood_swing_frequency -> codificación ordinal

withdrawal_symptoms, loss_of_other_interests, continued_despite_problems, eye_strain y back_neck_pain son binarias 

que mierda hacemos con mood_state???

In [20]:
X.describe(include='object')

,gender,game_genre,sleep_quality,sleep_disruption_frequency,academic_work_performance,mood_state,mood_swing_frequency
count,1000,1000,1000,1000,1000,1000,1000
unique,3,7,5,5,6,9,5
top,Male,MOBA,Fair,Often,Average,Normal,Sometimes
freq,647,156,293,221,246,172,214


#### Agrupar las clases a predecir para convertir el problema en un problema de clasificación binario:

La variable de salida tiene 4 categorías diferentes: Low, Moderate, High y Severe. Se juntarán Low con Moderate (formando Low Risk) y High con Severe (formando High Risk).

In [21]:
y = y.replace({
    'Low': 'Low Risk',
    'Moderate': 'Low Risk',
    'High': 'High Risk',
    'Severe': 'High Risk'
})

In [22]:
y.describe()

count         1000
unique           2
top       Low Risk
freq           704
Name: gaming_addiction_risk_level, dtype: object

#### Métrica de rendimiento:

chat si queremos ver el riesgo de volverse adicto utilizar recall por ejemplo ¿?¿?¿?

### **Pipeline automatizada**
#### Dataset $\rightarrow$ Imputación de valores perdidos $\rightarrow$ Transformación de tipos $\rightarrow$ KNN

#### _Imputación de valores perdidos_
Se observan las variables en las que es necesario imputar valores.

In [23]:
necesarias = np.array(variables_entrada)[X.isnull().sum() > 0]
print(necesarias)

['grades_gpa' 'work_productivity_score']


IMPUTACIÓN: variables: grades_gpa y work_productivity_score, ambas numéricas.

Grades GPA: 1 - 4

Work prod: 1 - 10

No están relacionadas en el sentido de que: si una falta falta otra. Hay ejemplos donde falta solo 1.

#### Técnicas de imputación: 
- Cold deck: No es posible creo yo, no hay algo que permita deducir bien las notas ni el productivity score
- De los métodos hot deck, probaría el del **vecino más cercano** (lo veo el más robusto)
- Otros que probaría: **media**, **mediana**... **regresión (multivar.???)** ??
- Imputación multiple lo veo fumadilla

In [24]:
#EJEMPLOS
transformer = ColumnTransformer(
    transformers = [("meanImputation", SimpleImputer(strategy='mean'), necesarias)], # para cambiar entre técnicas, cambiar mean por median
    remainder = 'passthrough' # el resto de columnas no hace nada con ellas, si pones drop las borra
)

transformer_2 = ColumnTransformer( # para que este funcione bien habría que, primero, transformar las variables categoricas a numericas.
    transformers = [("KnnImputation", KNNImputer(n_neighbors=5, weights='uniform'), necesarias)],
    remainder = 'passthrough'
)

transformer.set_output(transform="pandas") # asi sale un dataframe directamente esta chetado
transformer_2.set_output(transform="pandas")

X_imputados = transformer_2.fit_transform(X)
X_imputados

,KnnImputation__grades_gpa,KnnImputation__work_productivity_score,remainder__age,remainder__gender,remainder__daily_gaming_hours,remainder__game_genre,remainder__sleep_hours,remainder__sleep_quality,remainder__sleep_disruption_frequency,remainder__academic_work_performance,...,remainder__loss_of_other_interests,remainder__continued_despite_problems,remainder__eye_strain,remainder__back_neck_pain,remainder__weight_change_kg,remainder__exercise_hours_weekly,remainder__social_isolation_score,remainder__face_to_face_social_hours_weekly,remainder__monthly_game_spending_usd,remainder__years_gaming
0,1.250,5.0,17,Male,11.1,Mobile Games,3.7,Very Poor,Sometimes,Below Average,...,False,True,True,False,6.8,3.7,7,1.3,383.70,3
1,3.750,2.0,21,Male,3.0,MOBA,7.2,Fair,Rarely,Good,...,False,False,False,False,0.4,8.5,2,10.7,46.64,1
2,1.830,9.0,23,Male,7.6,FPS,4.4,Fair,Often,Poor,...,True,True,False,True,1.8,7.1,5,3.2,100.81,6
3,1.620,2.0,20,Female,7.2,RPG,5.1,Fair,Often,Poor,...,True,False,True,True,0.2,5.2,4,9.1,51.60,7
4,2.440,6.0,18,Male,6.8,Battle Royale,3.4,Poor,Never,Average,...,False,False,False,False,0.5,6.1,4,4.5,32.57,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,2.960,6.4,15,Female,8.6,Strategy,4.1,Very Poor,Always,Average,...,False,True,True,True,1.9,7.5,6,2.4,426.54,3
996,3.420,7.8,18,Male,5.4,MMO,6.5,Fair,Never,Excellent,...,False,False,True,False,2.1,7.7,1,10.9,83.71,7
997,2.886,3.0,23,Male,7.3,RPG,3.9,Insomnia,Rarely,Poor,...,True,False,False,True,0.5,8.1,5,6.7,88.60,5
998,1.030,3.4,18,Male,3.1,Strategy,8.2,Fair,Sometimes,Good,...,False,False,False,False,0.8,8.4,1,12.7,22.02,8


#### _Transformación de los tipos: categórica $\rightarrow$ numérica_
Una vez imputados los valores, para clasificar con KNN se requiere que las variables sean de tipo numérico. Por ello, es necesario transformar las variables categóricas a numéricas siguiendo diferentes técnicas de codificación.

**Codificación binaria**

gender, game_genre, mood_state, loss_of_other_interests, withdrawal_symptoms, eye_strain, back_neck_pain, continued_despite_problems

**Codificación ordinal**

sleep_quality, sleep_disruption_frequency, academic_work_performance, mood_swing_frequency


#### _Definición de los encoders_

In [25]:
def transformaBinario(df):
    transformed = df.apply(lambda x: x.astype(int) if x.dtype == 'bool' else x)
    return transformed
bool_to_int = FunctionTransformer(transformaBinario)

ord = ce.OrdinalEncoder(
    cols=['sleep_quality', 'sleep_disruption_frequency', 'academic_work_performance', 'mood_swing_frequency'],
    mapping = [
        {'col': 'sleep_quality', 'mapping': {'Insomnia': 1, 'Very Poor': 2, 'Poor': 3, 'Fair': 4, 'Good': 5}},
        {'col': 'sleep_disruption_frequency', 'mapping': {'Never': 1, 'Rarely': 2, 'Sometimes': 3, 'Often': 4, 'Always': 5}},
        {'col': 'academic_work_performance', 'mapping': {'Failing': 1, 'Poor': 2, 'Below Average': 3, 'Average': 4, 'Good': 5, 'Excellent': 6}},
        {'col': 'mood_swing_frequency', 'mapping': {'Never': 1, 'Rarely': 2, 'Sometimes': 3, 'Often': 4, 'Daily': 5}}
    ],
    handle_missing = 'return_nan',
    handle_unknown = 'return_nan' #revisar ... 
)

binary = ce.BinaryEncoder(
    cols = ['gender', 'game_genre', 'mood_state'],
    handle_missing = 'return_nan'
)

#### _Pipeline global 1_
En esta pipeline la transformación de las columnas es simple, pues se puede realizar en paralelo.

In [26]:
transformer_1 = ColumnTransformer(
    transformers = [("meanImputation", SimpleImputer(strategy='median'), necesarias), 
                    ("ordEncoder", ord, ['sleep_quality', 'sleep_disruption_frequency', 'academic_work_performance', 'mood_swing_frequency']),
                    ("binEncoder", binary, ['gender', 'game_genre', 'mood_state']),
                    ("boolEncoder", bool_to_int, ['loss_of_other_interests', 'withdrawal_symptoms', 'eye_strain', 'back_neck_pain', 'continued_despite_problems'])],
    remainder = 'passthrough'
)
pipeline_1 = Pipeline([('transformer', transformer_1),('modelo', neighbors.KNeighborsClassifier())])

#### _Pipeline global 2_
En este segundo caso, si se quiere imputar mediante KNN, hay que transformar antes las variables categóricas a numéricas.

In [27]:
transformer_2 = ColumnTransformer(
    transformers = [("ordEncoder", ord, ['sleep_quality', 'sleep_disruption_frequency', 'academic_work_performance', 'mood_swing_frequency']),
                    ("binEncoder", binary, ['gender', 'game_genre', 'mood_state']),
                    ("boolEncoder", bool_to_int, ['loss_of_other_interests', 'withdrawal_symptoms', 'eye_strain', 'back_neck_pain', 'continued_despite_problems'])],
    remainder = 'passthrough',
    verbose_feature_names_out = False
)
transformer_2.set_output(transform="pandas")

transformer_2_aux = ColumnTransformer(
    transformers = [("KnnImputation", KNNImputer(n_neighbors=5, weights='uniform'), necesarias)],
    remainder = 'passthrough'
)

pipeline_2 = Pipeline([('transformer_num', transformer_2), ('KNNimputation', transformer_2_aux),('modelo', neighbors.KNeighborsClassifier())])

In [28]:
# train, val, test: 80-10-10
aux_X, X_test, aux_y, y_test = train_test_split(X, y, test_size=0.1, stratify=y, random_state=123)
X_train, X_val, y_train, y_val =  train_test_split(aux_X, aux_y, stratify=aux_y, test_size=1/9, random_state=123)

print(y_train.size, y_val.size, y_test.size)

800 100 100


In [29]:
pipeline_1.fit(X_train, y_train)
y_pred1_val = pipeline_1.predict(X_val)
y_pred1_test = pipeline_1.predict(X_test)

accuracy1_val = accuracy_score(y_val, y_pred1_val) * 100
accuracy1_test = accuracy_score(y_test, y_pred1_test) * 100

print(f"Accuracy en Val: {accuracy1_val}%")
print(f"Accuracy en Test: {accuracy1_test}%")

Accuracy en Val: 81.0%
Accuracy en Test: 83.0%


In [30]:
pipeline_2.fit(X_train, y_train)
y_pred2_val = pipeline_2.predict(X_val)
y_pred2_test = pipeline_2.predict(X_test)

accuracy2_val = accuracy_score(y_val, y_pred2_val) * 100
accuracy2_test = accuracy_score(y_test, y_pred2_test) * 100

print(f"Accuracy en Val: {accuracy2_val}%")
print(f"Accuracy en Test: {accuracy2_test}%")

Accuracy en Val: 82.0%
Accuracy en Test: 83.0%
